In [0]:
activity_bronze = spark.table(
    "health_and_habit_guardian.bronze.activity"
)

In [0]:
from pyspark.sql import functions as F

# store all timestamps in UTC; fix the Spark session time zone so that
# _processed_at and any timestamp casts are written consistently in UTC
spark.conf.set("spark.sql.session.timeZone", "UTC")

# explicitly cast types in the Silver layer to estabilish a canonical schema

activity = (
    activity_bronze
    .withColumn(
        "steps",
        F.col("steps").cast("int")
    )
    .withColumn(
        "active_minutes",
        F.col("active_minutes").cast("int")
    )
    .withColumn(
        "calories_burned",
        F.col("calories_burned").cast("int")
    )
    .withColumn(
        "resting_hr",
        F.col("resting_hr").cast("int")
    )
)

In [0]:
# determine what constitutes as a valid activity record

activity = (
    activity
    .withColumn(
        "missing_required",
        (
            F.col("user_id").isNull() |
            F.col("date").isNull()
        )
    )
    .withColumn(
        "invalid_steps",
        F.col("steps").isNull() |
        (F.col("steps") < 0)
    )
    .withColumn(
        "invalid_active_minutes",
        F.col("active_minutes").isNull() |
        (F.col("active_minutes") < 0)
    )
    .withColumn(
        "invalid_calories",
        F.col("calories_burned").isNull() |
        (F.col("calories_burned") < 0)
    )
    .withColumn(
        "invalid_resting_hr",
        F.col("resting_hr").isNull() |
        (F.col("resting_hr") < 30) |
        (F.col("resting_hr") > 220)
    )
)

In [0]:
# add a quality status

activity = activity.withColumn(
    "_quality_status",
    F.when(
        F.col("missing_required") |
        F.col("invalid_steps") |
        F.col("invalid_active_minutes") |
        F.col("invalid_active_minutes") |
        F.col("invalid_calories") |
        F.col("invalid_resting_hr"),
        F.lit("INVALID")
    ).otherwise(F.lit("VALID"))
)

In [0]:
activity = activity.withColumn(
    "_processed_at",
    F.current_timestamp()
)

In [0]:
# if there are multiple users for the same user/day, keep the latest ingestion

from pyspark.sql.window import Window

window = Window.partitionBy(
    "user_id",
    "date"
).orderBy(
    F.col("_ingested_at").desc()
)

activity_deduped = (
    activity
    .withColumn(
        "_row_number",
        F.row_number().over(window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
# upsert the deduplicated + valid records to Silver;
# quarantine still sees every invalid record (including duplicates)

valid_activity = (
    activity_deduped
    .filter(F.col("_quality_status") == "VALID")
)

invalid_activity = (
    activity
    .filter(F.col("_quality_status") == "INVALID")
)

In [0]:
(
    invalid_activity
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "health_and_habit_guardian.silver.quarantine_activity"
    )
)

In [0]:
# idempotent upsert on the business key (user_id, date)
# so late-arriving data updates the existing Silver record

valid_activity.createOrReplaceTempView("activity_updates")

spark.sql("""
CREATE TABLE IF NOT EXISTS health_and_habit_guardian.silver.activity
USING DELTA
AS SELECT * FROM activity_updates WHERE 1 = 0
""")

# dedupe the source inside the MERGE so the upsert can never fail with
# [DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE]; latest ingestion wins
cols = ", ".join(valid_activity.columns)

spark.sql(f"""
MERGE INTO health_and_habit_guardian.silver.activity AS target
USING (
    SELECT {cols}
    FROM (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY user_id, date
            ORDER BY _ingested_at DESC NULLS LAST
        ) AS _rn
        FROM activity_updates
    )
    WHERE _rn = 1
) AS source
ON target.user_id = source.user_id AND target.date = source.date
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")


In [0]:
%run ./00_quality_utils


In [0]:
write_quality_summary(
    df=activity,
    table_name="activity",
    date_col="date",
    dedup_cols=["user_id", "date"],
)
